In [ ]:
!pip install -q -U transformers peft trl datasets accelerate
!pip uninstall -y torchao

In [ ]:
import os, glob
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

train_path = glob.glob("/kaggle/input/**/v2_train.json", recursive=True)[0]
val_path = glob.glob("/kaggle/input/**/v2_val.json", recursive=True)[0]
print(train_path, val_path)

ds = load_dataset("json", data_files={"train": train_path, "validation": val_path})
print(ds)

mid = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(mid)
model = AutoModelForCausalLM.from_pretrained(mid, dtype=torch.float32, device_map={"": 0})
print(next(model.parameters()).dtype, torch.cuda.device_count())

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
args = SFTConfig(output_dir="out", per_device_train_batch_size=4, gradient_accumulation_steps=4,
                 learning_rate=2e-4, num_train_epochs=2, logging_steps=5,
                 eval_strategy="steps", eval_steps=10, fp16=True, max_length=512, report_to="none")
trainer = SFTTrainer(model=model, args=args, train_dataset=ds["train"],
                     eval_dataset=ds["validation"], peft_config=lora, processing_class=tok)
trainer.train()

In [ ]:
trainer.model.save_pretrained("/kaggle/working/format-lora-v2")
tok.save_pretrained("/kaggle/working/format-lora-v2")

from huggingface_hub import login
login()   # paste your Hugging Face Write token when asked
trainer.model.push_to_hub("Meow07/qwen2.5-0.5b-format-lora-v2")
tok.push_to_hub("Meow07/qwen2.5-0.5b-format-lora-v2")